In [2]:
#!pip install lightning

  Using cached lightning-2.5.1.post0-py3-none-any.whl.metadata (39 kB)
  Using cached torch-2.7.0-cp39-cp39-win_amd64.whl.metadata (29 kB)
Using cached lightning-2.5.1.post0-py3-none-any.whl (819 kB)
Using cached torch-2.7.0-cp39-cp39-win_amd64.whl (212.4 MB)

  Attempting uninstall: torch

    Found existing installation: torch 1.13.1

   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
    Uninstalling torch-1.13.1:
   ---------------------------------------- 0/2 [torch]
      Successfully uninstalled torch-1.13.1
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   -----------------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sdmetrics 0.6.0 requires torch<2,>=1.8.0, but you have torch 2.7.0 which is incompatible.


In [1]:
import os
import json
import pickle
import random
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from lightning import seed_everything
from torchmetrics import AUROC, AveragePrecision, F1Score, MetricCollection, StatScores
from tqdm import trange

from pytrial.tasks.trial_outcome import HINT, SPOT
from pytrial.tasks.trial_outcome.data import TrialOutcomeDataset, TrialOutcomeDatasetBase

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score

C:\Users\Carol\anaconda3\envs\pytrial39\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
import torch.serialization
torch.serialization.add_safe_globals([np.core.multiarray._reconstruct])

In [3]:
class Args:
    model = "hint,spot,mlp,lr,xgb"
    phase = "I,II,III"
    no_bootstrap_test = False
    output_path = "results"
    n = 1
    world_size = 1
    rank = 0

args = Args()

In [4]:
def get_random_seed():
    return random.randint(np.iinfo(np.uint32).min, np.iinfo(np.uint32).max)

def load_custom_hint_dataframe(base_path, phase, split):
    csv_file = os.path.join(base_path, f"phase_{phase}_{split}.csv")
    criteria_file = os.path.join(base_path, "criteria", f"phase_{phase}_{split}.npy")
    table_file = os.path.join(base_path, "text_description", f"phase_{phase}_{split}.json")
    summary_file = os.path.join(base_path, "brief_summary", f"phase_{phase}_{split}.json")
    description_file = os.path.join(base_path, "drugbank", "druginfo_description.json")

    df = pd.read_csv(csv_file)

    # Rename 'smiles' to 'smiless' for compatibility with PyTrial
    if 'smiles' in df.columns:
        df.rename(columns={'smiles': 'smiless'}, inplace=True)

    for col in ['smiless', 'diseases', 'drugs']:
        if col not in df.columns:
            df[col] = 'None'

    if os.path.exists(criteria_file):
        criteria = np.load(criteria_file)
        if len(criteria) > len(df):
            criteria = criteria[:len(df)]
        elif len(criteria) < len(df):
            df = df.iloc[:len(criteria)]
        df['criteria'] = list(criteria)

    if 'enrollment' not in df.columns:
        df['enrollment'] = np.random.uniform(100, 1000, size=len(df))

    if 'nctid' not in df.columns:
        df['nctid'] = [f'NCT{phase}{split.upper()}{i:05d}' for i in range(len(df))]

    if os.path.exists(table_file):
        with open(table_file) as f:
            table_data = json.load(f)
        if isinstance(table_data, list):
            df['table'] = df.index.map(lambda i: table_data[i] if i < len(table_data) else "")
        else:
            df['table'] = df.index.map(lambda i: table_data.get(str(i), ""))

    if os.path.exists(summary_file):
        with open(summary_file) as f:
            summary_data = json.load(f)
        if isinstance(summary_data, list):
            df['summarization'] = df.index.map(lambda i: summary_data[i] if i < len(summary_data) else "")
        else:
            df['summarization'] = df.index.map(lambda i: summary_data.get(str(i), ""))

    if os.path.exists(description_file):
        with open(description_file) as f:
            drug_descriptions = json.load(f)
        def extract_description(drugs):
            try:
                drug_list = eval(drugs) if isinstance(drugs, str) else drugs
                return [drug_descriptions.get(drug, "This is a drug.") for drug in drug_list]
            except:
                return ["This is a drug."]
        df['description'] = df['drugs'].apply(extract_description)

    return df.dropna(subset=['label', 'criteria'])

In [5]:
def bootstrap_test(preds, target, metrics, bootstrap_num=20):
    results = []
    for _ in range(bootstrap_num):
        idx = torch.randint(0, len(target), (len(target),))
        cur_result = metrics(preds[idx], target[idx])
        results.append(cur_result)
        metrics.reset()

    final_result = {}
    for key in results[0]:
        values = torch.stack([r[key] for r in results])
        final_result[f"{key}_mean"] = values.float().mean().item()
        final_result[f"{key}_std"] = values.float().std().item()
    return final_result

In [6]:
def hint(datasets, metrics, data, no_bootstrap_test, *args, **kwargs):
    model = HINT(highway_num_layer=2, epoch=5, lr=1e-3, device="cpu")
    model.fit(datasets["train"], datasets["valid"])

    result = {}
    for split in ["valid", #"test"]:
        preds = model.predict(datasets[split])
        target_dict = data[split].set_index("nctid")["label"].to_dict()
        target = torch.tensor([target_dict[k[0]] for k in preds], dtype=torch.long)
        pred_scores = torch.tensor([k[1] for k in preds], dtype=torch.float32)
        result[split] = metrics(pred_scores, target) if no_bootstrap_test else bootstrap_test(pred_scores, target, metrics)
        metrics.reset()
    return result

def dummy_predict(model, X):
    return torch.tensor(model.predict_proba(X)[:, 1])

def traditional_model_fn(name):
    if name == 'lr':
        return LogisticRegression(max_iter=500)
    elif name == 'mlp':
        return MLPClassifier(hidden_layer_sizes=(64,), max_iter=300)
    elif name == 'xgb':
        return XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    else:
        raise ValueError("Unknown traditional model")

def run_traditional_model(datasets, metrics, data, no_bootstrap_test, model_name, *args, **kwargs):
    df_train = datasets["train"].data
    df_valid = datasets["valid"].data
    df_test = datasets["test"].data

    feature_cols = ['enrollment']
    X_train = df_train[feature_cols]
    y_train = df_train['label']
    X_test = df_test[feature_cols]
    y_test = df_test['label']

    model = traditional_model_fn(model_name)
    model.fit(X_train, y_train)

    preds = torch.tensor(model.predict_proba(X_test)[:, 1], dtype=torch.float32)
    target = torch.tensor(y_test.values, dtype=torch.float32)

    return {
        "test": metrics(preds, target) if no_bootstrap_test else bootstrap_test(preds, target, metrics),
        "valid": {}
    }

In [7]:
def spot(*args, **kwargs):
    kwargs['device'] = 'cpu'
    model = SPOT(seed=kwargs['seed'], output_dir=kwargs['output_path'], device='cpu')
    model.fit(kwargs['datasets']['train'], kwargs['datasets']['valid'])
    result = {}
    for split in ["valid", "test"]:
        preds = model.predict(kwargs['datasets'][split])
        pred_tensor = torch.tensor(preds['pred'][:, 0], dtype=torch.float32)
        target_tensor = torch.tensor(preds['label'], dtype=torch.float32)
        result[split] = kwargs['metrics'](pred_tensor, target_tensor.long()) if kwargs['no_bootstrap_test'] else bootstrap_test(pred_tensor, target_tensor, kwargs['metrics'])
        kwargs['metrics'].reset()
    shutil.rmtree(kwargs['output_path'])
    return result

model_map = defaultdict(lambda: run_traditional_model)
model_map['hint'] = hint
model_map['spot'] = spot


In [8]:
def fit_modal(model_name, phase, no_bootstrap_test, n, world_size, rank, output_path, *args, **kwargs):
    output_path = os.path.join(output_path, phase, model_name)
    os.makedirs(output_path, exist_ok=True)

    for i in trange(rank, n, world_size):
        seed = get_random_seed()
        seed_everything(seed)
        result = model_map[model_name](
            *args,
            **kwargs,
            no_bootstrap_test=no_bootstrap_test,
            seed=seed,
            model_name=model_name,
            output_path=os.path.join(output_path, str(i)),
        )
        result["seed"] = seed
        with open(os.path.join(output_path, f"{i}.pkl"), "wb") as f:
            pickle.dump(result, f)

In [9]:
def main():
    metrics = MetricCollection({
        "F1": F1Score("binary"),
        "ROC-AUC": AUROC("binary"),
        "PR-AUC": AveragePrecision("binary"),
        "STAT": StatScores("binary"),
    })

    hint_data_path = r"C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint"

    for phase in args.phase.split(","):
        data = {
            split: load_custom_hint_dataframe(hint_data_path, phase, split)
            for split in ["train", "valid", "test"]
        }

        for model in args.model.split(","):
            datasets = {
                split: TrialOutcomeDatasetBase(data[split]) if model == 'hint' else TrialOutcomeDataset(data[split])
                for split in ["train", "valid", "test"]
            }
            fit_modal(
                model,
                phase,
                args.no_bootstrap_test,
                args.n,
                args.world_size,
                args.rank,
                output_path=args.output_path,
                datasets=datasets,
                metrics=metrics,
                data=data,
            )

if __name__ == "__main__":
    main()

Seed set to 4019019599                                                                           | 0/1 [00:00<?, ?it/s]


| Uniform Initialization
| Uniform Initialization



%|                                                                                            | 0/5 [00:00<?, ?it/s]

                                                                                         | 0/170 [00:00<?, ?it/s]

                                                                                 | 1/170 [00:02<05:57,  2.11s/it]

epoch: 0, loss: 0.6270835995674133




                                                                                 | 2/170 [00:04<05:57,  2.13s/it]

...

epoch: 4, loss: 0.3127425014972687




100%|██████████████████████████████████████████████████████████████████████████████████| 1/1 [33:08<00:00, 1988.58s/it]
Seed set to 1346491743                                                                           | 0/1 [00:00<?, ?it/s]


Load pretrained Trial2Vec model from ./trial_search/pretrained_trial2vec


  0%|                                                                                            | 0/1 [00:02<?, ?it/s]


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.ndarray was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.ndarray])` or the `torch.serialization.safe_globals([numpy.ndarray])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weight

In [ ]:
import pandas as pd
import os
import pickle

def collect_results(output_dir, phases, models):
    table = []
    for phase in phases:
        for model in models:
            model_path = os.path.join(output_dir, phase, model, "0.pkl")
            if not os.path.exists(model_path):
                print(f"Missing: {model_path}")
                continue
            with open(model_path, "rb") as f:
                results = pickle.load(f)
            row = {
                "Method": model.upper(),
                "Phase": f"Phase {phase.upper()}",
                "F1": round(results["test"].get("F1_mean", 0) * 100, 1),
                "PR": round(results["test"].get("PR-AUC_mean", 0) * 100, 1),
                "ROC": round(results["test"].get("ROC-AUC_mean", 0) * 100, 1),
            }
            table.append(row)
    return pd.DataFrame(table)

# Example usage:
output_dir = "results"
phases = ["I", "II", "III"]
models = ["lr", "mlp", "xgb", "hint", "spot"]
df_summary = collect_results(output_dir, phases, models)

# Optional: pivot for cleaner view
pivot_df = df_summary.pivot(index="Method", columns="Phase", values=["PR", "F1", "ROC"])
pivot_df = pivot_df.swaplevel(axis=1).sort_index(axis=1)

# Show the table
print(pivot_df)

# Optional export
# pivot_df.to_csv("model_performance_summary.csv")


def main():
    metrics = MetricCollection({
        "F1": F1Score("binary"),
        "ROC-AUC": AUROC("binary"),
        "PR-AUC": AveragePrecision("binary"),
        "STAT": StatScores("binary"),
    })

    hint_data_path = r"C:\Users\Carol\Documents\Data Code and Deliverables\Data Group Part\Data\clinical-trial-outcome-prediction\Processed\hint"  # Replace with your actual path

    for phase in args.phase.split(","):
        data = {
            split: load_custom_hint_dataframe(hint_data_path, phase, split)
            for split in ["train", "valid", "test"]
        }

        for model in args.model.split(","):
            datasets = {
                split: TrialOutcomeDatasetBase(data[split]) if model == 'hint' else TrialOutcomeDataset(data[split])
                for split in ["train", "valid", "test"]
            }
            fit_modal(
                model,
                phase,
                args.no_bootstrap_test,
                args.n,
                args.world_size,
                args.rank,
                output_path=args.output_path,
                datasets=datasets,
                metrics=metrics,
                data=data,
            )

if __name__ == "__main__":
    main()
